In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from scipy.cluster.hierarchy import fcluster

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_subject_topk(subject_index=2):
    topk = np.load("top_k_abs.npy", allow_pickle=True).item()
    return topk[subject_index]

In [ ]:
def load_all_subjects_topk():
    topk = np.load("top_k_abs.npy", allow_pickle=True).item()
    return topk

In [ ]:
def get_top_k_keys(d, k, reverse=True):
    """
    Returns the top k keys in a dictionary that have the highest values.

    Parameters:
    d (dict): The input dictionary.
    k (int): The number of top keys to return.

    Returns:
    list: A list of the top k keys with the highest values.
    """
    # Sort the dictionary by values in descending order and get the top k keys
    top_k_keys = sorted(d, key=d.get, reverse=reverse)[:k]
    return top_k_keys

# Example usage
d = {'a': 10, 'b': 20, 'c': 15, 'd': 5, 'e': 25}
k = 3
print(get_top_k_keys(d, k))  # Output: ['e', 'b', 'c']

In [ ]:
def get_top_k_keys_all_subjects():
    topk = load_all_subjects_topk()
    topk_keys = {}
    for subject_index in topk.keys():
        topk_keys[subject_index] = get_top_k_keys(topk[subject_index], 60)
    return topk_keys


In [ ]:
top_k_all_subjects = get_top_k_keys_all_subjects()

In [ ]:
import pickle


def load_predicted_amplitude_for_subject(subject_index=2, rep=1):
    data_dir = f"/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/gradshap_explanations_rep_{rep}"
    file_path = os.path.join(data_dir, f"gradshap_data_subject_{subject_index}_rep_{rep}.npy")

    subject_data = np.load(file_path, allow_pickle=True).item()
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [ ]:
#channel_importance(explanations, ch_names)

In [ ]:
mne.set_log_level("ERROR")

In [ ]:
def get_common_channels(top_k_all_subjects):
    # Get the first subject's channel names as a set
    common_channels = set(top_k_all_subjects[1])
    
    # Intersect with each other subject's channels
    for subject_id in top_k_all_subjects.keys():
        subject_channels = set(top_k_all_subjects[subject_id])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(top_k_all_subjects.keys())} subjects")
    print("Common channels:", common_channels)
    
    return common_channels
common_channels = get_common_channels(top_k_all_subjects)

In [ ]:
def plot_agreement_topomap_importances(top1,top2, ch_names, info_subj1, agreement,subj1, subj2 ):
    common_channels= np.intersect1d(top1, top2)
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(3, 3))
    fig.suptitle(f' {subj1} vs {subj2}')

    agree_channels = np.zeros(len(ch_names))
    for ch in common_channels:
        agree_channels[ch_names.index(ch)] = 1
    mne.viz.plot_topomap(agree_channels, info_subj1, show=False, names=ch_names, 
                           axes=ax, image_interp="nearest")
    ax.set_title(f'agreement ratio: {agreement:.2f}')

    

In [ ]:
def load_subject_info(subject_index):

    file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
    epochs = mne.read_epochs(file_path)
    info_subj = epochs.info
    return info_subj

In [ ]:
def load_info_all_subjects():
    all_subjects_info = {}
    for subject_index in cfg.dataset.test_subject_indices:
        _, _, _, ch_names = load_predicted_amplitude_for_subject(subject_index)

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info
        all_subjects_info[subject_index] = {"ch_names": ch_names, "info_subj": info_subj}
    return all_subjects_info

# clustering functions

In [ ]:
def get_common_channels(all_subjects_data):
    # Get the first subject's channel names as a set
    first_subject = list(all_subjects_data.keys())[0]
    common_channels = set(all_subjects_data[first_subject]["ch_names"])
    
    # Intersect with each other subject's channels
    for subject_id in all_subjects_data.keys():
        subject_channels = set(all_subjects_data[subject_id]["ch_names"])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(all_subjects_data)} subjects")
    print("Common channels:", common_channels)
    
    return common_channels

In [ ]:
cfg = load_config()
all_subjects_info = load_info_all_subjects()

In [ ]:
def get_cluster_labels(linkage_matrix, n_clusters=4):
    """Extract cluster labels from linkage matrix"""

    return fcluster(linkage_matrix, n_clusters, criterion='maxclust')

In [ ]:
common_channels = get_common_channels(all_subjects_info)

# rank correlations

## gradshap

In [ ]:
channel_importance = np.load("all_subject_channel_importances_gradshap_rep_1.npy", allow_pickle=True).item()
channel_importance_abs = np.load("all_subject_channel_importances_gradshap_abs_rep_1.npy", allow_pickle=True).item()

In [ ]:
cfg = load_config()

In [ ]:
import scipy
def calculate_pairwise_correlations(importances1, importances2, common_channels):

    correlation_results = {}

        # Extract the top 10 channels for both subjects

            # Extract values for common channels only
    common_channel_values1 = np.array([importances1[ch] for ch in common_channels])
    common_channel_values2 = np.array([importances2[ch] for ch in common_channels])
            
    # Calculate correlations
    pearson_corr = np.corrcoef(common_channel_values1, common_channel_values2)[0,1]
    spearman_corr = scipy.stats.spearmanr(common_channel_values1, common_channel_values2).correlation
            
    # Store results
    correlation_results= {
                'pearson': pearson_corr,
                'spearman': spearman_corr}
  
    return correlation_results

In [ ]:
def plot_agreement_matrix_correlations_all_subjects(mean_correlations_all_subject_pairs, take_abs=False, label="gradshap"):
    # for a given frequency band and amplification factor, plot the correlation between all subjects
    # subjcet indicies are on rows and columns
    # the value in each cell is the mean correlation across all factors
    #
    correlation_matrix_pearson = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    correlation_matrix_spearman = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    subject_indices = list(cfg.dataset.test_subject_indices)                             
    for subj1,subj2 in itertools.combinations(subject_indices,2):
        if take_abs:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson'])
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson'])
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman'])
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman'])

        else:

            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson']
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['pearson']
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman']
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)]['spearman']
        

    # use the correlations matrices as distance metrics:
    distance_matrix_pearson = 1 - correlation_matrix_pearson
    distance_matrix_spearman = 1 - correlation_matrix_spearman
    np.save(f"distance_matrices/top_k_new_{label}_pearson.npy", distance_matrix_pearson)
    np.save(f"distance_matrices/top_k_new_{label}_spearman.npy", distance_matrix_spearman)


    
    fig,axs = plt.subplots(nrows=2, ncols=1, figsize=(25, 25))

    axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=-1, vmax=1)
    axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=-1, vmax=1)

    #cbar = fig.colorbar(im_spearman, ax=axs.ravel().tolist(), location='bottom', shrink=0.35)
    #cbar.set_label('Correlation')

    # Set axis labels
    for ax in axs:
        ax.set_xticks(np.arange(len(subject_indices)))
        ax.set_yticks(np.arange(len(subject_indices)))
        ax.set_xticklabels(subject_indices)
        ax.set_yticklabels(subject_indices)
        ax.set_xlabel('Subject indices')
        ax.set_ylabel('Subject indices')
        plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
    
    axs[0].set_title('Pearson correlation')
    axs[1].set_title('Spearman correlation')

    # Add text annotations
    for i in range(len(subject_indices)):
        for j in range(len(subject_indices)):
            axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                       ha="center", va="center", color="white")
            axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                       ha="center", va="center", color="white")



                                           

In [ ]:
def load_data_all_subjects():
    all_subjects_data = {}
    for subject_index in cfg.dataset.test_subject_indices:
        predictions, uncertainties, _, ch_names = load_predicted_amplitude_for_subject(subject_index)
        top_k = load_subject_topk(subject_index)
        

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info
        all_subjects_data[subject_index] = {"predictions": predictions, "uncertainties": uncertainties, "top_k": top_k, "ch_names": ch_names, "info_subj": info_subj}
    return all_subjects_data

In [ ]:
all_subjects_data = load_data_all_subjects()

In [ ]:
import itertools
rank_correlations_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    rank_correlations_all_pairs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(channel_importance[subject_index1], channel_importance[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs, take_abs=False, label="gradshap")

In [ ]:
rank_correlations_all_pairs_abs= {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    rank_correlations_all_pairs_abs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(channel_importance_abs[subject_index1], channel_importance_abs[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs, take_abs=False, label="gradshap_abs")

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import os

def cluster_and_visualize_correlations(rank_correlations_all_pairs,  correlation_type='spearman', take_abs=False, n_clusters=4, save_path=None, ax=None, treshold=None):

    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    
    # Get the list of unique subject IDs
    subject_ids = set()
    for pair in rank_correlations_all_pairs:
        subject_ids.add(pair[0])
        subject_ids.add(pair[1])
    subject_ids = sorted(list(subject_ids))
    n_subjects = len(subject_ids)
    
    # Create a mapping from subject ID to index
    subject_to_idx = {subj: idx for idx, subj in enumerate(subject_ids)}
    
    # Initialize the distance matrix with ones (maximum distance)
    distance_matrix = np.ones((n_subjects, n_subjects))
    np.fill_diagonal(distance_matrix, 0)  # Zero distance to self
    
    # Fill the distance matrix
    for pair, correlations in rank_correlations_all_pairs.items():
        i, j = subject_to_idx[pair[0]], subject_to_idx[pair[1]]
        corr_value = correlations[correlation_type]
        
        if take_abs:
            distance = 1 - abs(corr_value)
        else:
            distance = 1 - corr_value
            
        distance_matrix[i, j] = distance
        distance_matrix[j, i] = distance  # Matrix is symmetric
    
    # Convert the distance matrix to a condensed form for linkage
    condensed_matrix = squareform(distance_matrix)
    
    # Compute the linkage matrix
    linkage_matrix = linkage(condensed_matrix, method='ward')
    
    # Get cluster assignments
    cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    
    # Create figure and axis if ax is not provided
    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 4))
    
    # Create color map for clusters
    color_palette = cm.rainbow(np.linspace(0, 1, n_clusters))
    cluster_colors = {i+1: color_palette[i] for i in range(n_clusters)}
    
    cluster_color_map = {}
    for i, label in enumerate(cluster_labels):
        cluster_color_map[str(subject_ids[i])] = cluster_colors[label]
    
    # Plot dendrogram
    dendrogram(
        linkage_matrix,
        labels=subject_ids,
        ax=ax,
        leaf_rotation=90,
        leaf_font_size=10,
        color_threshold=treshold
    )
    
    # Color the leaves based on cluster
    #for i, label in zip(range(n_subjects), ax.get_xticklabels()):
    #    label.set_color(cluster_color_map[label.get_text()])
    
    #ax.set_title(f'Hierarchical Clustering ({n_clusters} clusters)\n{correlation_type.capitalize()} Correlation - {freq_band.capitalize()} Band')
    ax.set_xlabel('Subject ID', fontsize=18)
    ax.set_ylabel('Distance', fontsize=18)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, fontsize=16)
    ax.set_yticklabels(ax.get_yticks(), fontsize=16)

    # Format y-tick labels to show 2 decimal places
    yticks = ax.get_yticks()
    ax.set_yticklabels([f"{y:.2f}" for y in yticks], fontsize=16)
    # Save results if path is provided
    if save_path and fig is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(f"{save_path}_dendrogram.png", dpi=300, bbox_inches='tight')
        np.save(f"{save_path}_distance_matrix.npy", distance_matrix)
    
    fig.savefig(f"hierarchical_gradshap_spearman.png", dpi=300)

    return fig, cluster_labels, distance_matrix, subject_ids


In [ ]:
def plot_cluster_topomap(importances_all_subjects, cluster_labels, subject_ids, common_channels, normalize=False):         #   general idea for the subjects in a cluster, aggregate their values of median_differences and plot the topomap
    n_clusters = len(np.unique(cluster_labels))
    n_subjects = len(subject_ids)
    ch_names = all_subjects_data[1]["ch_names"]
    fig,axs = plt.subplots(nrows=1, ncols=n_clusters, figsize=(16, 4))
    #fig.suptitle('Topomap of Median Differences for Each Cluster')
    info = all_subjects_data[1]["info_subj"]
    for cluster_id in range(1, n_clusters + 1):
   
  
        # Find subjects in this cluster
        cluster_subjects = [subj for subj, label in zip(subject_ids, cluster_labels) if label == cluster_id]
        importance_channels_cluster = np.zeros(len(ch_names))
        for subject in cluster_subjects:
            subject_sum = 0
            importance_per_channel = importances_all_subjects[subject]
            median_diff = importance_per_channel
            if normalize:
                for ch in common_channels:
                    subject_sum += np.abs(median_diff[ch])

            for ch in common_channels:
                if normalize:
                    importance_channels_cluster[ch_names.index(ch)] += median_diff[ch] / subject_sum
                else:
                    importance_channels_cluster[ch_names.index(ch)] += median_diff[ch]
            # think about dividing results by number of subjects in the cluster 
     
     
        
        
        if n_clusters >1:
            axs[cluster_id-1].set_title(f'Cluster {cluster_id}, N: {len(cluster_subjects)}', fontsize=18)
            mne.viz.plot_topomap(importance_channels_cluster, info, axes=axs[cluster_id-1], show=False, names=ch_names)
        else:
            axs.set_title(f'Cluster {cluster_id}', fontsize=18)
            mne.viz.plot_topomap(importance_channels_cluster, info, axes=axs, show=False, names=ch_names)
    
    fig.savefig(f"topomap_cluster_gradshap_spearman.png", dpi=300, bbox_inches='tight')


    

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster, cophenet
from sklearn.metrics import silhouette_score

def evaluate_n_clusters(distance_matrix, method='average', max_k=10):

# Store results
    results = {}


    Z = linkage(distance_matrix, method=method)
    silhouette_scores = []

    within_cluster_dists = []
    dunn_indices = []
    
    for k in range(2, max_k + 1):
        labels = fcluster(Z, k, criterion='maxclust')
        silhouette = silhouette_score(distance_matrix, labels, metric='precomputed')

        
        # Compute within-cluster sum of distances
        within_dist = 0
        for cluster in np.unique(labels):
            indices = np.where(labels == cluster)[0]
            if len(indices) > 1:
                within_dist += np.sum(distance_matrix[np.ix_(indices, indices)])
        within_cluster_dists.append(within_dist)
        
        # Calculate Dunn Index
        max_intra_cluster_dist = 0
        min_inter_cluster_dist = float('inf')
        
        # Find max intra-cluster distance
        for cluster in np.unique(labels):
            indices = np.where(labels == cluster)[0]
            if len(indices) > 1:
                cluster_dists = distance_matrix[np.ix_(indices, indices)]
                max_dist = np.max(cluster_dists)
                max_intra_cluster_dist = max(max_intra_cluster_dist, max_dist)
        
        # Find min inter-cluster distance
        for i, cluster1 in enumerate(np.unique(labels)):
            indices1 = np.where(labels == cluster1)[0]
            for cluster2 in np.unique(labels)[i+1:]:
                indices2 = np.where(labels == cluster2)[0]
                inter_dists = distance_matrix[np.ix_(indices1, indices2)]
                min_dist = np.min(inter_dists)
                min_inter_cluster_dist = min(min_inter_cluster_dist, min_dist)
        
        # Calculate Dunn Index (avoid division by zero)
        if max_intra_cluster_dist > 0:
            dunn_idx = min_inter_cluster_dist / max_intra_cluster_dist
        else:
            dunn_idx = float('inf')
        
        dunn_indices.append(dunn_idx)
        silhouette_scores.append(silhouette)

        

    
    results[method] = {
        'silhouette': silhouette_scores,
        'within_cluster_dist': within_cluster_dists,
        'dunn_index': dunn_indices
    }

    return results


In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_all_pairs_abs, correlation_type='spearman', take_abs=False, n_clusters=5, save_path=None)
plot_cluster_topomap(channel_importance_abs, cluster_labels, subject_ids, common_channels)
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_all_pairs, correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None, treshold=1.7)
plot_cluster_topomap(channel_importance, cluster_labels, subject_ids, common_channels)
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(rank_correlations_all_pairs, correlation_type='spearman', take_abs=False, n_clusters=1, save_path=None, treshold=1.4)
plot_cluster_topomap(channel_importance, cluster_labels, subject_ids, common_channels)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

{'average': {'silhouette': [0.18751627721776687,
   0.1561438938369289,
   0.1463865907256193,
   0.1553635703309983,
   0.13696969129828462],
  'within_cluster_dist': [383.13372296902395,
   368.1133839859731,
   342.23775569842195,
   336.1574517825833,
   331.5739333722969],
  'dunn_index': [0.4538050806321125,
   0.4538050806321125,
   0.48652711923210357,
   0.48652711923210357,
   0.48652711923210357]}}

## saliency

In [ ]:
channel_importance_saliency = np.load("all_subject_channel_importances_saliency.npy", allow_pickle=True).item()
channel_importance_saliency_abs = np.load("all_subject_channel_importances_saliency_abs.npy", allow_pickle=True).item()

In [ ]:
cfg = load_config()

In [ ]:
saliency_rank_correlations_all_pairs_abs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    saliency_rank_correlations_all_pairs_abs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(channel_importance_saliency_abs[subject_index1], channel_importance_saliency_abs[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs_abs, take_abs=False, label="saliency_abs")

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(saliency_rank_correlations_all_pairs_abs, correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None)
plot_cluster_topomap(channel_importance_saliency_abs, cluster_labels, subject_ids, common_channels)
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)

In [ ]:
saliency_rank_correlations_all_pairs= {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    saliency_rank_correlations_all_pairs[(subject_index1, subject_index2)] = calculate_pairwise_correlations(channel_importance_saliency[subject_index1], channel_importance_saliency[subject_index2], common_channels)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_all_pairs, take_abs=False, label="saliency")

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(saliency_rank_correlations_all_pairs, correlation_type='spearman', take_abs=False, n_clusters=1, save_path=None)
plot_cluster_topomap(channel_importance_saliency, cluster_labels, subject_ids, common_channels)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(saliency_rank_correlations_all_pairs, correlation_type='spearman', take_abs=False, n_clusters=2, save_path=None, treshold=2.4)
plot_cluster_topomap(channel_importance_saliency, cluster_labels, subject_ids, common_channels)
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(saliency_rank_correlations_all_pairs, correlation_type='spearman', take_abs=False, n_clusters=3, save_path=None, treshold=1.7)
plot_cluster_topomap(channel_importance_saliency, cluster_labels, subject_ids, common_channels)
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channels, band_name="delta", normalize=True)

In [ ]:

_, cluster_labels, distance_matrix, subject_ids = cluster_and_visualize_correlations(saliency_rank_correlations_all_pairs, correlation_type='spearman', take_abs=False, n_clusters=5, save_path=None, treshold=1.5)
plot_cluster_topomap(channel_importance_saliency, cluster_labels, subject_ids, common_channels)
#plot_cluster_topomap(median_diff_per_channel_all_subjects, cluster_labels, subject_ids, common_channe

In [ ]:
evaluate_n_clusters(distance_matrix, method='average', max_k=6)